# Stage 6 — Regional Clustering  (data analysis, no prediction)

Motivation: Stage 4's regional forest showed the heat→SCS signal is regional (strongest in the
North) and Stage 5 found production is a continuum, not discrete animal types. The dataset now
carries `macro_area` / `Region` / `Province`. Question here: **is the geography a real structure?**
Do farms/regions form groups by their production + climate profile, and does that grouping recover
the North/Central/South split — or is region just a climate-exposure gradient?

1. Regional overview — coverage, imbalance (Campania+Lazio dominate).
2. Region-level profiles — interpretable mean trait+climate table per region.
3. Region-level clustering — Ward dendrogram + K-Means on region profiles; compare to macro_area.
4. Farm-level clustering (n=316) — no PCA; does the farm profile recover macro_area (ARI)?
   Climate-only vs production-only: which drives the geography?
5. Regional heat-response clustering — per-region within-animal summer THI slopes (milk & SCS),
   reproduce the Stage-4 regional forest, then group regions into heat-vulnerable types.
6. Season split — does farm structure shift summer vs winter.

Anchors: Stage 4 regional_forest (North SCS +0.41/+10, 30d), Trapanese continuum (Stage 5).
Checkpoint: climate (THI) separates macro_area strongly; North = coolest + highest SCS-heat slope.

In [1]:
import pandas as pd, numpy as np, json, warnings
from pathlib import Path
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, davies_bouldin_score
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")
SEED = 42

def find_root():
    cands = [Path("Thesis_Data"), Path("../../Thesis_Data")]
    try: cands.append(Path(__file__).resolve().parents[2] / "Thesis_Data")
    except NameError: pass
    for c in cands:
        if c.exists(): return c.resolve().parent
    raise SystemExit("no Thesis_Data")
ROOT = find_root(); OUT = ROOT / "codes" / "pipeline" / "outputs"
FIG = OUT / "figures"; TAB = OUT / "tables"; FIG.mkdir(exist_ok=True, parents=True); TAB.mkdir(exist_ok=True, parents=True)
def rule(m): print("\n" + "=" * 78 + f"\n{m}\n" + "=" * 78)
def verdict(n, ok, d=""): print(f"  [{'MATCH' if ok else 'CHECK'}] {n}" + (f"  --  {d}" if d else ""))
def sil(X, lab): return round(silhouette_score(X, lab), 3) if len(set(lab)) > 1 else np.nan
def best_k(X, ks=range(2, 7)):
    b = (None, -1, None)
    for k in ks:
        if k >= len(X): break
        lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(X)
        s = sil(X, lab)
        if s > b[1]: b = (k, s, lab)
    return b

base = pd.read_parquet(OUT / "analysis_base.parquet")
rule("LOADED")
print(f"  records={len(base):,} | animals={base.Animal_ID.nunique():,} | farms={base.Farm_Code.nunique()} "
      f"| regions={base.Region.nunique()} | macro_areas={base.macro_area.nunique()}")


LOADED
  records=1,620,204 | animals=90,532 | farms=316 | regions=16 | macro_areas=3


## 6.1 Regional overview — coverage & imbalance
The panel is heavily South-weighted (Campania) with a much smaller Northern tail. Any regional
clustering has to be read with this imbalance in mind: small regions = noisy profiles.

In [2]:
rule("6.1  REGIONAL OVERVIEW")
cov = (base.groupby(["macro_area", "Region"])
       .agg(farms=("Farm_Code", "nunique"), animals=("Animal_ID", "nunique"),
            records=("milk_kg", "size"))
       .reset_index().sort_values("records", ascending=False))
cov["rec_share_%"] = (100 * cov.records / cov.records.sum()).round(1)
cov.to_csv(TAB / "region_coverage.csv", index=False)
print(cov.to_string(index=False))
print("\n  macro_area totals:")
print(base.groupby("macro_area").agg(farms=("Farm_Code", "nunique"),
      animals=("Animal_ID", "nunique"), records=("milk_kg", "size")).to_string())


6.1  REGIONAL OVERVIEW
macro_area                Region  farms  animals  records  rec_share_%
     South              Campania    179    50043   863661         53.3
   Central                 Lazio     91    24861   449054         27.7
     South                Puglia     11     3219    65566          4.0
     North             Lombardia      4     2668    58686          3.6
     North              Piemonte      3     2108    41815          2.6
     South            Basilicata      6     1992    41588          2.6
     South              Calabria      4     1744    34351          2.1
     North Friuli Venezia Giulia      3     1075    22509          1.4
     North                Veneto      4     1095    18838          1.2
     South               Sicilia      4      864     9566          0.6
     South                Molise      1      294     7070          0.4
   Central               Toscana      2      349     4754          0.3
   Central                Marche      1      173     

## 6.2 Region-level trait + climate profiles (interpretable, no PCA)
One row per region: mean production, quality, herd structure and climate exposure. This is the
feature table the region clustering runs on.

In [3]:
rule("6.2  REGION-LEVEL PROFILES")
def heatwave_frac(s): return float((s >= 78).mean())  # THI_1_max >= 78 = heat-stress day (buffalo)
reg = (base.groupby("Region")
       .agg(farms=("Farm_Code", "nunique"), records=("milk_kg", "size"),
            milk_kg=("milk_kg", "mean"), fat_p=("fat_p", "mean"), protein_p=("protein_p", "mean"),
            SCS=("SCS", "mean"), ECM=("ECM", "mean"),
            THI_avg=("THI_1_avg", "mean"), THI_max=("THI_1_max", "mean"),
            parity=("parity", "mean"), AFC_mo=("true_AFC_days", lambda s: s.mean() / 30.44))
       .join(base.groupby("Region").THI_1_max.apply(heatwave_frac).rename("heatday_frac")))
reg["macro_area"] = base.groupby("Region").macro_area.first()
reg = reg.round(3)
reg.to_csv(TAB / "region_profiles.csv")
CLUS_FEATS = ["milk_kg", "fat_p", "protein_p", "SCS", "THI_avg", "THI_max", "heatday_frac", "parity", "AFC_mo"]
print(reg[["macro_area", "farms", "records"] + CLUS_FEATS].to_string())


6.2  REGION-LEVEL PROFILES
                      macro_area  farms  records  milk_kg  fat_p  protein_p    SCS  THI_avg  THI_max  heatday_frac  parity  AFC_mo
Region                                                                                                                            
Abruzzo                    South      1      186    8.807  7.926      4.933  1.991   56.702   66.851         0.237   1.000  31.836
Basilicata                 South      6    41588   10.180  8.131      4.779  3.600   57.933   67.937         0.260   2.660  34.226
Calabria                   South      4    34351    9.504  7.972      4.613  3.194   62.335   68.601         0.242   2.814  34.630
Campania                   South    179   863661    9.294  8.081      4.694  3.258   61.008   69.186         0.270   2.650  37.787
Emilia Romagna             North      1      387    8.514  7.858      4.623  3.257   56.617   65.085         0.225   2.667  39.168
Friuli Venezia Giulia      North      3    22509    9.4

## 6.3 Region-level clustering — Ward dendrogram + K-Means (does it recover macro_area?)
Only well-sampled regions (>=2 farms AND >=5000 records) — a 1-farm region is one herd, not a
region. Small N (points = regions) so the dendrogram is the honest tool; K-Means is a cross-check.

In [4]:
rule("6.3  REGION-LEVEL CLUSTERING")
regC = reg[(reg.farms >= 2) & (reg.records >= 5000)].copy()
print(f"  clustering {len(regC)} well-sampled regions: {list(regC.index)}")
Xr = StandardScaler().fit_transform(regC[CLUS_FEATS])
Z = linkage(Xr, method="ward")
for k in (2, 3, 4):
    lab = fcluster(Z, k, criterion="maxclust")
    ari = adjusted_rand_score(regC.macro_area, lab)
    print(f"  Ward cut k={k}: silhouette={sil(Xr, lab)}  ARI vs macro_area={round(ari, 3)}  "
          f"sizes={np.bincount(lab)[1:].tolist()}")
k_reg, sil_reg, lab_reg = best_k(Xr, ks=range(2, 6))
regC["region_cl"] = fcluster(Z, 3, criterion="maxclust")
print(f"\n  K-Means best k={k_reg} (silhouette={sil_reg})")
print("\n  Ward k=3 clusters vs macro_area:")
print(pd.crosstab(regC.region_cl, regC.macro_area).to_string())
prof3 = regC.groupby("region_cl")[["milk_kg", "SCS", "THI_avg", "THI_max", "heatday_frac"]].mean().round(2)
prof3["regions"] = regC.groupby("region_cl").apply(lambda d: ", ".join(d.index))
print("\n  cluster profiles:"); print(prof3.to_string())


6.3  REGION-LEVEL CLUSTERING
  clustering 10 well-sampled regions: ['Basilicata', 'Calabria', 'Campania', 'Friuli Venezia Giulia', 'Lazio', 'Lombardia', 'Piemonte', 'Puglia', 'Sicilia', 'Veneto']
  Ward cut k=2: silhouette=0.307  ARI vs macro_area=0.449  sizes=[5, 5]
  Ward cut k=3: silhouette=0.309  ARI vs macro_area=0.321  sizes=[5, 4, 1]
  Ward cut k=4: silhouette=0.313  ARI vs macro_area=0.443  sizes=[5, 3, 1, 1]

  K-Means best k=4 (silhouette=0.313)

  Ward k=3 clusters vs macro_area:
macro_area  Central  North  South
region_cl                        
1                 1      0      4
2                 0      3      1
3                 0      1      0

  cluster profiles:
           milk_kg   SCS  THI_avg  THI_max  heatday_frac                                        regions
region_cl                                                                                              
1             9.68  3.10    60.69    68.94          0.27  Basilicata, Calabria, Campania, Lazio, Puglia


## 6.4 Farm-level clustering (n=316, no PCA) — is geography recoverable?
One row per farm. Cluster on the full profile, then on climate-only and production-only feature
blocks, and measure how well each recovers macro_area (Adjusted Rand Index). Tells us whether the
North/South split is a *climate* fact or a *production* fact.

In [5]:
rule("6.4  FARM-LEVEL CLUSTERING")
farm = (base.groupby("Farm_Code")
        .agg(records=("milk_kg", "size"),
             milk_kg=("milk_kg", "mean"), fat_p=("fat_p", "mean"), protein_p=("protein_p", "mean"),
             SCS=("SCS", "mean"), THI_avg=("THI_1_avg", "mean"), THI_max=("THI_1_max", "mean"),
             parity=("parity", "mean"), AFC_mo=("true_AFC_days", lambda s: s.mean() / 30.44))
        .join(base.groupby("Farm_Code").THI_1_max.apply(heatwave_frac).rename("heatday_frac")))
farm["macro_area"] = base.groupby("Farm_Code").macro_area.first()
farm["Region"] = base.groupby("Farm_Code").Region.first()
farm = farm[farm.records >= 200].dropna()   # drop tiny farms with unstable means
print(f"  farms clustered: {len(farm)}")
blocks = {
    "full":       ["milk_kg", "fat_p", "protein_p", "SCS", "THI_avg", "THI_max", "heatday_frac", "parity", "AFC_mo"],
    "climate":    ["THI_avg", "THI_max", "heatday_frac"],
    "production": ["milk_kg", "fat_p", "protein_p", "SCS"],
}
frows = []
for name, feats in blocks.items():
    X = StandardScaler().fit_transform(farm[feats])
    k, s, lab = best_k(X, ks=range(2, 7))
    ari3 = adjusted_rand_score(farm.macro_area, KMeans(3, n_init=10, random_state=SEED).fit_predict(X))
    frows.append({"block": name, "best_k": k, "silhouette": round(s, 3),
                  "ARI_k=bestk": round(adjusted_rand_score(farm.macro_area, lab), 3),
                  "ARI_k=3_vs_macroarea": round(ari3, 3), "DBI": round(davies_bouldin_score(X, lab), 3)})
    if name == "full":
        farm["farm_cl"] = lab; k_full = k
ff = pd.DataFrame(frows); ff.to_csv(TAB / "farm_cluster_blocks.csv", index=False)
print(ff.to_string(index=False))
print("\n  full-profile clusters vs macro_area:")
print(pd.crosstab(farm.farm_cl, farm.macro_area).to_string())
fp = farm.groupby("farm_cl")[blocks["full"]].mean().round(2)
fp["n_farms"] = farm.farm_cl.value_counts()
print("\n  farm-cluster profiles:"); print(fp.to_string())


6.4  FARM-LEVEL CLUSTERING
  farms clustered: 298
     block  best_k  silhouette  ARI_k=bestk  ARI_k=3_vs_macroarea   DBI
      full       3       0.166        0.076                 0.076 1.978
   climate       2       0.588        0.165                 0.088 0.584
production       4       0.242        0.005                 0.016 1.212

  full-profile clusters vs macro_area:
macro_area  Central  North  South
farm_cl                          
0                 5     14     12
1                28      0     61
2                55      1    122

  farm-cluster profiles:
         milk_kg  fat_p  protein_p   SCS  THI_avg  THI_max  heatday_frac  parity  AFC_mo  n_farms
farm_cl                                                                                          
0           8.70   8.15       4.71  3.44    55.98    63.88          0.16    2.51   39.25       31
1           8.80   7.54       4.73  2.84    62.02    70.97          0.33    1.74   37.90       89
2           9.25   8.02       4.7

## 6.5 Regional heat-response clustering — reproduce Stage-4 forest, then group regions
Per region: within-animal, DIM-controlled summer slope of milk & SCS on THI_anom (per +10). Same
spec as Stage 4/5. Then cluster regions on (milk-slope, SCS-slope) to name heat-response types.

In [6]:
rule("6.5  REGIONAL HEAT-RESPONSE")
Sm = base[base.is_summer_warmhalf].copy(); Sm["DIM2"] = Sm.DIM.astype(float) ** 2
def region_slope(sub, trait, min_animals=200):
    d = sub[[trait, "THI_anom", "DIM", "DIM2", "Animal_ID", "Farm_Code"]].dropna().copy()
    if d.Animal_ID.nunique() < min_animals: return None
    g = d.groupby("Animal_ID")
    for c in [trait, "THI_anom", "DIM", "DIM2"]:
        d[c + "_w"] = d[c] - g[c].transform("mean")
    m = smf.ols(f"{trait}_w ~ THI_anom_w + DIM_w + DIM2_w - 1", data=d)
    # farm-clustered SE needs >=2 farms; single-farm regions fall back to heteroskedasticity-robust
    if d.Farm_Code.nunique() >= 2:
        r = m.fit(cov_type="cluster", cov_kwds={"groups": d.Farm_Code})
    else:
        r = m.fit(cov_type="HC1")
    return round(10 * r.params["THI_anom_w"], 4), round(10 * 1.96 * r.bse["THI_anom_w"], 4), int(d.Animal_ID.nunique())
hrows = []
for rg in Sm.Region.unique():
    sub = Sm[Sm.Region == rg]
    row = {"Region": rg, "macro_area": sub.macro_area.iloc[0], "n_animals_summer": sub.Animal_ID.nunique()}
    ok = True
    for t in ["milk_kg", "SCS"]:
        res = region_slope(sub, t)
        if res is None: ok = False; break
        row[f"{t}_per10"], row[f"{t}_ci95"] = res[0], res[1]
    if ok: hrows.append(row)
hh = pd.DataFrame(hrows).sort_values("SCS_per10", ascending=False)
hh.to_csv(TAB / "regional_heat_slopes.csv", index=False)
print(hh.to_string(index=False))
# cluster regions on (milk slope, SCS slope)
Xh = StandardScaler().fit_transform(hh[["milk_kg_per10", "SCS_per10"]])
k_h, s_h, lab_h = best_k(Xh, ks=range(2, 5))
hh["heat_cl"] = lab_h
print(f"\n  heat-response clusters k={k_h} (silhouette={s_h}):")
print(hh.groupby("heat_cl")[["milk_kg_per10", "SCS_per10"]].mean().round(3).to_string())
print("  regions per heat cluster:")
for c in sorted(hh.heat_cl.unique()):
    print(f"    cluster {c}: {', '.join(hh[hh.heat_cl == c].Region)}")


6.5  REGIONAL HEAT-RESPONSE
               Region macro_area  n_animals_summer  milk_kg_per10  milk_kg_ci95  SCS_per10  SCS_ci95
               Veneto      North              1082        -0.3565        0.3239     0.3436    0.2851
               Molise      South               285        -0.2316        0.2265     0.2594    0.1094
              Toscana    Central               346         0.6263        0.4671     0.2078    0.7482
              Sicilia      South               852        -0.3091        0.2481     0.1946    0.5747
             Calabria      South              1718        -0.2108        0.2920     0.1790    0.2796
             Piemonte      North              2071        -0.2613        0.0440     0.1146    0.1642
                Lazio    Central             24543        -0.0964        0.1283     0.1121    0.1185
            Lombardia      North              2639        -0.0584        0.2842     0.0282    0.0895
Friuli Venezia Giulia      North              1066        -0.0

## 6.6 Season split — does farm structure shift summer vs winter?

In [7]:
rule("6.6  SEASON SPLIT (farm clustering)")
for name, mask in [("summer", base.is_summer_warmhalf), ("winter", ~base.is_summer_warmhalf)]:
    fm = base[mask].groupby("Farm_Code").agg(
        milk_kg=("milk_kg", "mean"), SCS=("SCS", "mean"),
        THI_avg=("THI_1_avg", "mean"), THI_max=("THI_1_max", "mean")).dropna()
    ma = base[mask].groupby("Farm_Code").macro_area.first().reindex(fm.index)
    X = StandardScaler().fit_transform(fm)
    k, s, lab = best_k(X, ks=range(2, 6))
    ari = adjusted_rand_score(ma, KMeans(3, n_init=10, random_state=SEED).fit_predict(X))
    print(f"  {name}: farms={len(fm)}  best k={k}  silhouette={s}  ARI(k=3) vs macro_area={round(ari, 3)}")


6.6  SEASON SPLIT (farm clustering)
  summer: farms=315  best k=2  silhouette=0.258  ARI(k=3) vs macro_area=0.031
  winter: farms=316  best k=3  silhouette=0.402  ARI(k=3) vs macro_area=0.152


## 6.7 Literature / cross-stage checkpoint

In [8]:
rule("6.7  CHECKPOINT")
ari_full = ff.loc[ff.block == "full", "ARI_k=3_vs_macroarea"].iloc[0]
ari_clim = ff.loc[ff.block == "climate", "ARI_k=3_vs_macroarea"].iloc[0]
ari_prod = ff.loc[ff.block == "production", "ARI_k=3_vs_macroarea"].iloc[0]
sil_clim = ff.loc[ff.block == "climate", "silhouette"].iloc[0]
sil_prod = ff.loc[ff.block == "production", "silhouette"].iloc[0]
verdict("Climate exposure forms cleaner farm groups than production traits", sil_clim > sil_prod,
        f"climate silhouette={sil_clim} vs production {sil_prod}")
verdict("Geography tracks CLIMATE more than production (a climate gradient, not a production type)",
        ari_clim >= ari_prod, f"climate ARI={ari_clim} vs production ARI={ari_prod}")
north_scs = hh.loc[hh.macro_area == "North", "SCS_per10"]
verdict("Stage-4 reproduced: North has among the strongest SCS heat slope",
        len(north_scs) > 0 and north_scs.max() >= hh.SCS_per10.median(),
        f"North SCS_per10={north_scs.round(3).tolist()} (median {round(hh.SCS_per10.median(), 3)})")
summary = {
    "regions_total": int(base.Region.nunique()), "regions_wellsampled": int(len(regC)),
    "farms_clustered": int(len(farm)), "farm_best_k": int(k_full),
    "ARI_full": float(ari_full), "ARI_climate": float(ari_clim), "ARI_production": float(ari_prod),
    "silhouette_climate": float(sil_clim), "silhouette_production": float(sil_prod),
    "region_heat_clusters": int(k_h),
    "region_SCS_slopes": hh.set_index("Region").SCS_per10.to_dict(),
}
(TAB / "stage6_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("\n  summary:", json.dumps(summary, indent=2, default=str))


6.7  CHECKPOINT
  [MATCH] Climate exposure forms cleaner farm groups than production traits  --  climate silhouette=0.588 vs production 0.242
  [MATCH] Geography tracks CLIMATE more than production (a climate gradient, not a production type)  --  climate ARI=0.088 vs production ARI=0.016
  [MATCH] Stage-4 reproduced: North has among the strongest SCS heat slope  --  North SCS_per10=[0.344, 0.115, 0.028, 0.024] (median 0.113)

  summary: {
  "regions_total": 16,
  "regions_wellsampled": 10,
  "farms_clustered": 298,
  "farm_best_k": 3,
  "ARI_full": 0.076,
  "ARI_climate": 0.088,
  "ARI_production": 0.016,
  "silhouette_climate": 0.588,
  "silhouette_production": 0.242,
  "region_heat_clusters": 3,
  "region_SCS_slopes": {
    "Veneto": 0.3436,
    "Molise": 0.2594,
    "Toscana": 0.2078,
    "Sicilia": 0.1946,
    "Calabria": 0.179,
    "Piemonte": 0.1146,
    "Lazio": 0.1121,
    "Lombardia": 0.0282,
    "Friuli Venezia Giulia": 0.0239,
    "Puglia": 0.0066,
    "Basilicata": -0.0035

## 6.8 Figures — regional structure

In [9]:
rule("6.8  FIGURES")
# Fig 6.1: region dendrogram, leaves colored/labelled by macro_area
fig, ax = plt.subplots(figsize=(11, 5))
lbl = [f"{r} [{regC.loc[r, 'macro_area'][0]}]" for r in regC.index]
dendrogram(Z, labels=lbl, ax=ax, leaf_rotation=90, color_threshold=0.7 * Z[:, 2].max())
ax.set_title("6.1  Region dendrogram (Ward) — leaf tag = macro_area [N/C/S]")
ax.set_ylabel("Ward distance"); plt.tight_layout(); plt.savefig(FIG / "s6_region_dendrogram.png", dpi=130); plt.show()

# Fig 6.2: farms on climate (THI_avg) × production (milk), colored by macro_area
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
col = {"North": "steelblue", "Central": "seagreen", "South": "firebrick"}
for m, c in col.items():
    d = farm[farm.macro_area == m]
    ax[0].scatter(d.THI_avg, d.milk_kg, c=c, s=22, alpha=.7, label=m)
ax[0].set_xlabel("mean THI_1_avg (climate exposure)"); ax[0].set_ylabel("mean milk_kg")
ax[0].set_title("6.2  Farms: climate × production (color = macro_area)"); ax[0].legend(); ax[0].grid(alpha=.3)
sc = ax[1].scatter(farm.THI_avg, farm.milk_kg, c=farm.farm_cl, cmap="viridis", s=22, alpha=.7)
ax[1].set_xlabel("mean THI_1_avg"); ax[1].set_ylabel("mean milk_kg")
ax[1].set_title(f"6.2  Same farms colored by K-Means cluster (k={k_full})"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "s6_farm_clusters.png", dpi=130); plt.show()

# Fig 6.3: regional heat forest — SCS & milk slope per +10 THI anomaly, ordered by SCS slope
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, (t, cc) in zip(ax, [("SCS", "firebrick"), ("milk_kg", "steelblue")]):
    s = hh.sort_values(f"{t}_per10"); yv = range(len(s))
    a.errorbar(s[f"{t}_per10"], yv, xerr=s[f"{t}_ci95"], fmt="o", capsize=3, color=cc)
    a.axvline(0, c="grey", lw=.8); a.set_yticks(list(yv))
    a.set_yticklabels([f"{r} [{m[0]}]" for r, m in zip(s.Region, s.macro_area)])
    a.set_title(f"6.3  {t}: within-animal summer heat slope per +10 THI-anom"); a.set_xlabel("Δ per +10"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "s6_regional_heat_forest.png", dpi=130); plt.show()

# Fig 6.4: heat-response map — milk slope vs SCS slope, colored by heat cluster, sized by n_animals
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(hh.milk_kg_per10, hh.SCS_per10, c=hh.heat_cl, cmap="coolwarm",
                s=30 + hh.n_animals_summer / hh.n_animals_summer.max() * 300, alpha=.75, edgecolor="k")
for _, r in hh.iterrows():
    ax.annotate(r.Region, (r.milk_kg_per10, r.SCS_per10), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.axhline(0, c="grey", lw=.8); ax.axvline(0, c="grey", lw=.8)
ax.set_xlabel("milk heat slope (Δ per +10)"); ax.set_ylabel("SCS heat slope (Δ per +10)")
ax.set_title("6.4  Regional heat-response map (top-left = heat-vulnerable: milk↓ & SCS↑)")
ax.grid(alpha=.3); plt.tight_layout(); plt.savefig(FIG / "s6_heat_response_map.png", dpi=130); plt.show()
print("  wrote s6_region_dendrogram / s6_farm_clusters / s6_regional_heat_forest / s6_heat_response_map")
rule("STAGE 6 COMPLETE")


6.8  FIGURES
  wrote s6_region_dendrogram / s6_farm_clusters / s6_regional_heat_forest / s6_heat_response_map

STAGE 6 COMPLETE
